# Limpieza y estandarización de catalogos

In [1]:
import pandas as pd
from pandas.api.types import CategoricalDtype
from itertools import product
from numpy import nan
from helper_pandas import df_to_dict, simplify_string

Cargar Bitacoras onomásticas y funciones auxiliares

In [11]:
# Break and create dataframe
def break_and_simplify(string, sort=False):
    string = string.replace(' x ','+').replace(' y ','+')
    string = string.split('+')
    string = [simplify_string(s, str_case=None) for s in string if len(s)>0]
    if sort: string = sorted(string, sort=True)
    return string

def is_missing(s, dic):
    return ~s.isin(dic.values()) & s.notna()

def highlight_missing(s, dic, color='yellow'):
    miss = is_missing(s,dic)
    return ['background-color: yellow' if v else '' for v in miss]

def join_series(series, join_str='+', sort=False):
    series = series.dropna()
    if sort:  series = series.drop_duplicates().sort_values()
    series = join_str.join(series.to_list())
    if len(series)==0: series = nan
    return series

def first_n_to_text(s, n=5, join_str=','):
    d = s.head(n).to_list()
    d = join_str.join(d)
    return d

from joblib import load

in_file = 'data_clean/HBO_clean.pkl'
with open(in_file, 'rb') as f:
     df = load(f)  
df = df.reset_index()
df.tail()

,ID,Numero_progresivo_transcrito,Nombre_completo_transcrito,Primer_apellido,Segundo_apellido,Nombres_propios,Fecha_transcrito,Fecha_estandar,Fecha_año,Fecha_mes,...,Diagnostico_clasificacion,Procedencia_transcrito,Procedencia_estandar,Procedencia_extendido,Procedencia_alcaldia,Numero_acta_transcrito,Expediente_SEMEFO_transcrito,Bitacora_ingresos,Foja_transcrito,Observaciones
96839,BO_1982_07489,S-D,placenta,NaN,NaN,NaN,1982-06-05,1982-06-05,1982.0,6.0,...,NaN,15a,GAM-2,Coord Territorial 2 Gustavo A. Madero (Col Ara...,Gustavo A. Madero,960,3079,semefo_df_bo_1982,156,NaN
96840,BO_1982_07490,S-D,5 dedos del pie derecho de desconocido,NaN,NaN,NaN,1982-06-05,1982-06-05,1982.0,6.0,...,NaN,32a,COY-2,Coord Territorial 2 Coyoacan (Col Romero de Te...,Coyoacan,950,3060,semefo_df_bo_1982,156,NaN
96841,BO_1982_07491,S-D,dedo de desconocido,NaN,NaN,NaN,1982-11-19,1982-11-19,1982.0,11.0,...,NaN,32a,COY-2,Coord Territorial 2 Coyoacan (Col Romero de Te...,Coyoacan,2005,6389,semefo_df_bo_1982,156,NaN
96842,BO_1982_07492,S-D,4 dedos de desconocido,NaN,NaN,NaN,1982-11-28,1982-11-28,1982.0,11.0,...,NaN,27a,XO-2,Coord Territorial 2 Xochimilco (Barrio de San ...,Xochimilco,959,6528,semefo_df_bo_1982,156,NaN
96843,BO_1982_07493,S-D,osamenta de desconocido,NaN,NaN,NaN,1982-10-11,1982-10-11,1982.0,10.0,...,NaN,9a,MH-1,Coord Territorial 1 Miguel Hidalgo – Hospital ...,Miguel Hidalgo,4193,5629,semefo_df_bo_1982,156,no se recibio necropsia. es una osamenta.


## Revisar diagnosticos

**Metodología**
* Los diagnosticos se dividieron usando '+'
* Se determinaron las partes de diagnosticos y se resolvieron por frecuencia
    - AA es anemia o asfixia 
* Se determinaron diagnosticos equivalentes
    - AOV y AOVA
* Se genero diagnosticos simplificados estandar y clasifico
* Se reviso los diagnosticos no solubles y se propuso separaciones alternativas, por patron de espacio, x y similitud de string
    - AnxEmbarazo -> An+Embarazo
* Proceso iterativo con los faltantes

**Faltantes**
* P, E, CS, CQ, A, L, PTE, T, IP, M, H, I, EH, IA, MP

In [6]:
df['Diagnostico_transcrito'].nunique()

3709

### cat_diag_extendido.csv
* Diagnostico | Extendido | Clasificacion | Frecuencia

In [5]:
df_ext = pd.read_csv('catalogos/cat_diag_extendido.csv')
df_ext

display(df_ext.head())

# Revisar llaves unicas
df_ext['Diagnostico'].nunique()
print("Llaves unicas {}/{}".format( df_ext['Diagnostico'].nunique(), df_ext.shape[0]))

# Revisar llaves unicas
df_ext['Diagnostico'].nunique()
print("Extend unicas {}/{}".format( df_ext['Extendido'].nunique(), df_ext.shape[0]))

# Faltantes
print('Faltantes:', df_ext.isna().sum())

dic_ext = df_to_dict(df_ext[['Diagnostico','Extendido']])
len( dic_ext )

dic_cla = df_to_dict(df_ext[['Diagnostico','Clasificacion']])
len( dic_cla )

,Diagnostico,Extendido,Clasificacion
0,AQ,amputacion quirurgica,Amputación
1,AQXT,amputacion quirurgica por traumatismo,Amputación
2,AT,amputacion traumatica,Amputación
3,AQXTMi,amputacion quirurgica por traumatismo miembro,Amputación
4,AQxTrat,amputacion quirurgica por tratamiento,Amputación


Llaves unicas 244/244
Extend unicas 244/244
Faltantes: Diagnostico       0
Extendido         0
Clasificacion    79
dtype: int64


244

### cat_diag_sustitucion.xlsx
* Diagnostico_transcrito | Diagnostico_estandar | Freq | Npartes | Nfalta | Partes coloreadas | IDs

In [9]:
df_sus = pd.read_excel('catalogos/cat_diag_sustitucion.xlsx')
# quitar remplazos iguales
df_sus = df_sus[df_sus['Diagnostico_transcrito']!=df_sus['Diagnostico_estandar']]
display( df_sus.head() )

# Revisar transcrito unicas
print("Llaves unicas {}/{}".format( df_sus['Diagnostico_transcrito'].nunique(), df_sus.shape[0]))

# Revisar faltantes
print("Transcrito faltantes {}/{}".format( df_sus['Diagnostico_transcrito'].isna().sum(), df_sus.shape[0]))
print("Transcrito estandar {}/{}".format( df_sus['Diagnostico_estandar'].isna().sum(), df_sus.shape[0]))

# Diccionario
dic_sus = df_to_dict(df_sus)
#agregar llaves extendido
dic_sus.update( {d:d for d in df_ext['Diagnostico'].unique()} )
print("Diccionario diagnosticos", len( dic_sus ))

,Diagnostico_transcrito,Diagnostico_estandar,Diagnostico_extendido,Freq,Npartes,Nfalta,IDs,Diagnostico_clasificacion
6,HPAFPC,HPAFC,herida arma fuego craneo,833,1,0,"BO_1974_00037,BO_1974_00104,BO_1974_00162,BO_1...",Herida por arma de fuego
10,Quemaduras,Quem,quemaduras,697,1,0,"BO_1974_00034,BO_1974_00051,BO_1974_00089,BO_1...",Causas externas de morbilidad y de mortalidad
13,TCE+BN,BN+TCE,bronconeumonia + traumatismo craneo encefalico,569,2,0,"BO_1974_00053,BO_1974_00315,BO_1974_00347,BO_1...",Enfermedades del sistema respiratorio + Trauma
15,AA,AXA,asfixia por ahorcamiento,514,1,0,"BO_1974_00025,BO_1974_00070,BO_1974_00100,BO_1...",Causas externas de morbilidad y de mortalidad
17,AQxT,AQXT,amputacion quirurgica por traumatismo,466,1,0,"BO_1975_06705,BO_1975_06706,BO_1975_06710,BO_1...",Amputación


Llaves unicas 3020/3020
Transcrito faltantes 0/3020
Transcrito estandar 0/3020
Diccionario diagnosticos 3264


### Regenerar tablas de frecuencia

Esto parte del diagnostico simplificado, aunque utiliza el separado para sustituciones complejas. Corre bajo tu propio riesgo

Separar y regenerar diagnostico transcrito

In [12]:
# Agrupar diagnostico y poner frecuencia y ejemplos
data = df.groupby('Diagnostico_transcrito').agg(
                      Freq=('ID', 'size'), 
                      IDs=('ID', first_n_to_text)
                 )
data = data.sort_values('Freq', ascending=False).reset_index()

# Remplazar complejas
data['Remplazo'] = data['Diagnostico_transcrito'].replace(dic_sus)
# Contar partes de cada diagnostico
data['Npartes'] = data['Remplazo'].str.count('\+') + 1

# Separar en partes
target_col = data['Remplazo']
sep = [break_and_simplify(s) for s in target_col.to_list()]
sep = pd.DataFrame(sep, index=target_col).fillna(nan)
sep_col = [f'Sep{i+1}' for i in range(sep.shape[1])]
sep.columns = sep_col
sep = sep.replace(dic_sus) # ! estandariza las partes usando sustitucion
sep

# Contar faltantes
sep['Nfalta'] = sep.apply(is_missing, dic=dic_sus).sum(axis=1)
sep['Diagnostico_estandar'] = sep[sep_col].apply(join_series, axis=1) # ! regenerar el estandar
sep = sep.reset_index()

# Unir 
data = pd.concat([data, sep], axis=1, join='inner').drop('Remplazo', axis=1)

# Agregar extendido
ext_col = list()
for col in sep_col:
    new_col = col.replace('Sep','Ext')
    data[new_col] = data[col].replace(dic_ext)
    ext_col.append(new_col)
data['Diagnostico_extendido'] = data[ext_col].apply(join_series, join_str=' ', axis=1)

# Ordenar
data = data[['Diagnostico_transcrito','Diagnostico_estandar','Diagnostico_extendido',
             'Freq','Npartes','Nfalta'] +sep_col +ext_col +['IDs']]

# Colorear faltantes
data_color = data.style.apply(highlight_missing, subset=sep_col, dic=dic_sus)
data_color.to_excel('catalogos/cat_diag_sustitucion_revisar.xlsx', index=False)
data_color
data['Nfalta'].sum()

511

In [13]:
data

,Diagnostico_transcrito,Diagnostico_estandar,Diagnostico_extendido,Freq,Npartes,Nfalta,Sep1,Sep2,Sep3,Sep4,Sep5,Ext1,Ext2,Ext3,Ext4,Ext5,IDs
0,S-D,S-D,sin datos,55878,1,0,S-D,NaN,NaN,NaN,NaN,sin datos,NaN,NaN,NaN,NaN,"BO_1968_00001,BO_1968_00002,BO_1968_00003,BO_1..."
1,TM,TM,traumatismo multiple,7505,1,0,TM,NaN,NaN,NaN,NaN,traumatismo multiple,NaN,NaN,NaN,NaN,"BO_1974_00019,BO_1974_00020,BO_1974_00026,BO_1..."
2,TCE,TCE,traumatismo craneo encefalico,5939,1,0,TCE,NaN,NaN,NaN,NaN,traumatismo craneo encefalico,NaN,NaN,NaN,NaN,"BO_1974_00018,BO_1974_00021,BO_1974_00030,BO_1..."
3,BN,BN,bronconeumonia,1826,1,0,BN,NaN,NaN,NaN,NaN,bronconeumonia,NaN,NaN,NaN,NaN,"BO_1974_00005,BO_1974_00008,BO_1974_00011,BO_1..."
4,CVG,CVG,congestion visceral generalizada,1287,1,0,CVG,NaN,NaN,NaN,NaN,congestion visceral generalizada,NaN,NaN,NaN,NaN,"BO_1974_00151,BO_1974_00165,BO_1974_00166,BO_1..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3704,HC+PN,HC+PN,hemorragia cerebral o herida cortante pleuro p...,1,2,0,HC,PN,NaN,NaN,NaN,hemorragia cerebral o herida cortante,pleuro pneumonia,NaN,NaN,NaN,BO_1975_04916
3705,HC+SH,HC+SH,hemorragia cerebral o herida cortante shock,1,2,0,HC,SH,NaN,NaN,NaN,hemorragia cerebral o herida cortante,shock,NaN,NaN,NaN,BO_1977_02656
3706,HC+TMI,HC+TMi,hemorragia cerebral o herida cortante traumati...,1,2,0,HC,TMi,NaN,NaN,NaN,hemorragia cerebral o herida cortante,traumatismo miembro,NaN,NaN,NaN,BO_1975_02086
3707,HC+Tb,HC+TB,hemorragia cerebral o herida cortante tubercul...,1,2,0,HC,TB,NaN,NaN,NaN,hemorragia cerebral o herida cortante,tuberculosis,NaN,NaN,NaN,BO_1978_04557


### Diagnosticos sencillos por frecuencia

In [14]:
sep = []
for col in sep_col :
    d = data[[col,'Freq']]
    d.columns = ['Sep', 'Freq']
    sep.append(d)
sep = pd.concat(sep, ignore_index=True)

sep = sep.dropna()
sep = sep.groupby('Sep').sum().reset_index() \
         .sort_values('Freq', ascending=False)
sep['Miss'] = is_missing(sep['Sep'], dic=dic_sus)

sep = sep.sort_values(['Freq','Sep'], ascending=[False, True])

sep.to_csv('catalogos/diag_partes_freq.csv', index=False)
sep

,Sep,Freq,Miss
508,S-D,55901,False
600,TM,8020,False
549,TCE,6945,False
95,BN,5328,False
125,CH,1535,False
...,...,...,...
673,intox. al,1,True
674,micosis,1,True
676,quemaduras,1,True
677,sd,1,True


In [15]:
sep[sep['Miss']==True].head(25)

,Sep,Freq,Miss
460,PTE,17,True
1,A,11,True
143,CQ,11,True
522,T,11,True
363,IP,7,True
622,TQT,7,True
146,CS,6,True
152,CUY,5,True
347,IA,5,True
412,MP,5,True


## Revisar procedencias

### cat_proc_extendido.csv
* Procedencia | Extendido | Alcaldia | Frecuencia

In [26]:
df_ext = pd.read_csv('catalogos/cat_proc_extendido.csv')
df_ext

display(df_ext.tail())

# Revisar llaves unicas
print("Llaves unicas {}/{}".format( df_ext['Procedencia'].nunique(), df_ext.shape[0]))

# Revisar llaves unicas
print("Extend unicas {}/{}".format( df_ext['Extendido'].nunique(), df_ext.shape[0]))

# Faltantes
print('Faltantes:', df_ext.isna().sum())

dic_ext = df_to_dict(df_ext[['Procedencia','Extendido']])
len( dic_ext )

dic_cla = df_to_dict(df_ext[['Procedencia','Alcaldia']])
len( dic_cla )


,Procedencia,Extendido,Alcaldia
56,VC-2,Coord Territorial 2 Venustiano Carranza (Col M...,Venustiano Carranza
57,XO-1,Coord Territorial 1 Xochimilco - Reclusorio Su...,Xochimilco
58,XO-2,Coord Territorial 2 Xochimilco (Barrio de San ...,Xochimilco
59,S-D,Sin datos,NaN
60,VO,Villa Obregon,Alvaro Obregon


Llaves unicas 61/61
Extend unicas 61/61
Faltantes: Procedencia    0
Extendido      0
Alcaldia       2
dtype: int64


61

### cat_diag_sustitucion.xlsx
* Procedencia_transcrito | Procedencia_estandar | Freq | Npartes | Nfalta | Partes coloreadas | IDs

In [28]:
df_sus = pd.read_excel('catalogos/cat_proc_sustitucion.xlsx')
display( df_sus.head() )

# Revisar transcrito unicas
print("Llaves unicas {}/{}".format( df_sus['Procedencia_transcrito'].nunique(), df_sus.shape[0]))

# Revisar faltantes
print("Transcrito faltantes {}/{}".format( df_sus['Procedencia_transcrito'].isna().sum(), df_sus.shape[0]))
print("Transcrito estandar {}/{}".format( df_sus['Procedencia_estandar'].isna().sum(), df_sus.shape[0]))

# Diccionario
dic_sus = df_to_dict(df_sus)
#agregar llaves extendido
dic_sus.update( {d:d for d in df_ext['Procedencia'].unique()} )
print("Diccionario procedencias", len( dic_sus ))

,Procedencia_transcrito,Procedencia_estandar,Procedencia_extendido,Freq,Npartes,Nfalta,IDs,Procedencia_alcaldia
0,S-D,S-D,Sin datos,25779,1,0,"BO_1968_00001,BO_1968_00002,BO_1968_00003,BO_1...",NaN
1,32a,COY-2,Coord Territorial 2 Coyoacan (Col Romero de Te...,2073,1,0,"BO_1977_00143,BO_1977_00175,BO_1977_00204,BO_1...",Coyoacan
2,33a,HB,Hospital Balbuena (Col Jardin Balbuena),1968,1,0,"BO_1977_00177,BO_1977_00184,BO_1977_00185,BO_1...",Venustiano Carranza
3,37a,CR,Cruz Roja (Col Polanco),1701,1,0,"BO_1977_00182,BO_1977_00196,BO_1977_00201,BO_1...",Miguel Hidalgo
4,32,COY-2,Coord Territorial 2 Coyoacan (Col Romero de Te...,1562,1,0,"BO_1977_05169,BO_1978_01205,BO_1980_00002,BO_1...",Coyoacan


Llaves unicas 845/845
Transcrito faltantes 0/845
Transcrito estandar 62/845
Diccionario procedencias 884


### Regenerar tablas de frecuencia

Esto parte del procedenciar simplificado, aunque utiliza el separado para sustituciones complejas. Corre bajo tu propio riesgo

Separar y regenerar procedencia transcrito

In [30]:
# Agrupar diagnostico y poner frecuencia y ejemplos
data = df.groupby('Procedencia_transcrito').agg(
                      Freq=('ID', 'size'), 
                      IDs=('ID', first_n_to_text)
                 )
data = data.sort_values('Freq', ascending=False).reset_index()

# Remplazar complejas
data['Remplazo'] = data['Procedencia_transcrito'].replace(dic_sus)
# Contar partes de cada procedencia
data['Npartes'] = data['Remplazo'].str.count('\+') + 1

# Separar en partes
target_col = data['Remplazo']
sep = [break_and_simplify(s) for s in target_col.to_list()]
sep = pd.DataFrame(sep, index=target_col).fillna(nan)
sep_col = [f'Sep{i+1}' for i in range(sep.shape[1])]
sep.columns = sep_col
sep = sep.replace(dic_sus) # ! estandariza las partes usando sustitucion
sep

# Contar faltantes
sep['Nfalta'] = sep.apply(is_missing, dic=dic_sus).sum(axis=1)
sep['Procedencia_estandar'] = sep[sep_col].apply(join_series, axis=1) # ! regenerar el estandar
sep = sep.reset_index()

# Unir 
data = pd.concat([data, sep], axis=1, join='inner').drop('Remplazo', axis=1)

# Agregar extendido
ext_col = list()
for col in sep_col:
    new_col = col.replace('Sep','Ext')
    data[new_col] = data[col].replace(dic_ext)
    ext_col.append(new_col)
data['Procedencia_extendido'] = data[ext_col].apply(join_series, join_str=' ', axis=1)

# Ordenar
data = data[['Procedencia_transcrito','Procedencia_estandar','Procedencia_extendido',
             'Freq','Npartes','Nfalta'] +sep_col +ext_col +['IDs']]

# Colorear faltantes
data_color = data.style.apply(highlight_missing, subset=sep_col, dic=dic_sus)
data_color.to_excel('catalogos/cat_proc_sustitucion_revisar.xlsx', index=False)
data_color
data['Nfalta'].sum()

AttributeError: 'float' object has no attribute 'replace'

### Procedencias sencillos por frecuencia

In [ ]:
sep = []
for col in sep_col :
    d = data[[col,'Freq']]
    d.columns = ['Sep', 'Freq']
    sep.append(d)
sep = pd.concat(sep, ignore_index=True)

sep = sep.dropna()
sep = sep.groupby('Sep').sum().reset_index() \
         .sort_values('Freq', ascending=False)
sep['Miss'] = is_missing(sep['Sep'], dic=dic_sus)

sep = sep.sort_values(['Freq','Sep'], ascending=[False, True])

sep.to_csv('catalogos/proc_partes_freq.csv', index=False)
sep

In [ ]:
sep[sep['Miss']==True].head(25)

Estandarizar a extendido

In [ ]:
sep['Extendido'] = sep['Sep'].map(dic_ext)
sep

In [ ]:
sep[sep['Extendido'].isna()].head(25)

### Diagnosticos astringente con alcaldia

In [ ]:
data_filter = data[['Procedencia_transcrito'] + sep_col]
cla_col = [col.replace('Sep','Cla') for col in sep_col]

for sep, ext, cla in zip(sep_col, ext_col, cla_col):
    data[sep] = data[sep].map(dic_sus)
    data[ext] = data[sep].map(dic_ext)
    data[cla] = data[sep].map(dic_cla)

data['Procedencia_estandar'] = data[sep_col].apply(join_series, sort=True, axis=1)
data['Procedencia_extendido'] = data[ext_col].apply(join_series, sort=True, join_str=' + ', axis=1)
data['Procedencia_alcaldia'] = data[cla_col].apply(join_series, sort=True, join_str=' + ', axis=1)

data = data.drop(sep_col+ext_col+cla_col, axis=1)
data = data[data['Procedencia_estandar'].notna()]
data.to_csv('catalogos/cat_proc_sustitucion.csv', index=False)
data[ data['Nfalta']>0 ]


Done